In [0]:
# GOLD FACT ORDERS
# CONFORMED FACT TABLE
from pyspark.sql import functions as F
from pyspark.sql.window import Window
CATALOG = "retail_demo"
RAW_SCHEMA = "raw"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"
BASE_PATH = (
    "/Volumes/retail_demo/raw/retail_files/"
    "retail_delta_project"
)
print(f"Catalog       : {CATALOG}")
print(f"Silver Schema : {SILVER_SCHEMA}")
print(f"Gold Schema   : {GOLD_SCHEMA}")
print(f"Base Path     : {BASE_PATH}")

Catalog       : retail_demo
Silver Schema : silver
Gold Schema   : gold
Base Path     : /Volumes/retail_demo/raw/retail_files/retail_delta_project


In [0]:
# CREATE / VERIFY STORE DIMENSION

store_source = spark.table(
    f"{CATALOG}.{SILVER_SCHEMA}.silver1_stores_clean"
)
print("Source rows:", store_source.count())
store_dim = (
    store_source
    .select(
        "store_id",
        "store_name",
        "city",
        "region",
        "status"
    )
    .dropDuplicates(["store_id"])
)
print("Dimension rows:", store_dim.count())

store_dim.printSchema()

display(store_dim.limit(10))

(
    store_dim
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{CATALOG}.{SILVER_SCHEMA}.dim_store"
    )
)

print("\nStore dimension created successfully.")

display(
    spark.sql(f"""
        SELECT
            COUNT(*) AS total_rows,
            COUNT(DISTINCT store_id) AS distinct_stores
        FROM {CATALOG}.{SILVER_SCHEMA}.dim_store
    """)
)

Source rows: 75
Dimension rows: 75
root
 |-- store_id: string (nullable = true)
 |-- store_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- status: string (nullable = true)



store_id,store_name,city,region,status
S001,Store_1,Jaipur,Online,active
S002,Store_2,Pune,West,active
S003,Store_3,Ahmedabad,South,closed
S004,Store_4,Ahmedabad,North,active
S005,Store_5,Chennai,East,active
S006,Store_6,Kolkata,East,closed
S007,Store_7,Hyderabad,Online,active
S008,Store_8,Chennai,North,closed
S009,Store_9,Bengaluru,West,active
S010,Store_10,Gurugram,East,active



Store dimension created successfully.


total_rows,distinct_stores
75,75


In [0]:
#  VERIFY GOLD SOURCE TABLES

gold_source_tables = {
    "Orders Silver": (
        f"{CATALOG}.{SILVER_SCHEMA}.silver1_orders_clean"
    ),
    "Customer SCD2": (
        f"{CATALOG}.{SILVER_SCHEMA}.dim_customer_scd2"
    ),
    "Product SCD2": (
        f"{CATALOG}.{SILVER_SCHEMA}.dim_product_scd2"
    ),
    "Store Dimension": (
        f"{CATALOG}.{SILVER_SCHEMA}.dim_store"
    )
}
for label, table_name in gold_source_tables.items():

    try:
        df = spark.table(table_name)

        print(f"\n {label}")
        print(f"  Table : {table_name}")
        print(f"  Rows  : {df.count():,}")

    except Exception as e:
        print(f"\n {label}")
        print(f"  Table : {table_name}")
        print(f"  ERROR : {str(e)[:200]}")
print("\n--- ORDERS ---")
spark.table(
    f"{CATALOG}.{SILVER_SCHEMA}.silver1_orders_clean"
).printSchema()

print("\n--- CUSTOMER SCD2 ---")
spark.table(
    f"{CATALOG}.{SILVER_SCHEMA}.dim_customer_scd2"
).printSchema()

print("\n--- PRODUCT SCD2 ---")
spark.table(
    f"{CATALOG}.{SILVER_SCHEMA}.dim_product_scd2"
).printSchema()

print("\n--- STORE ---")
spark.table(
    f"{CATALOG}.{SILVER_SCHEMA}.dim_store"
).printSchema()


 Orders Silver
  Table : retail_demo.silver.silver1_orders_clean
  Rows  : 5,716

 Customer SCD2
  Table : retail_demo.silver.dim_customer_scd2
  Rows  : 3,067

 Product SCD2
  Table : retail_demo.silver.dim_product_scd2
  Rows  : 980

 Store Dimension
  Table : retail_demo.silver.dim_store
  Rows  : 75

--- ORDERS ---
root
 |-- order_id: string (nullable = true)
 |-- order_ts: timestamp (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: decimal(18,2) (nullable = true)
 |-- discount_pct: double (nullable = true)
 |-- gross_amount: decimal(18,2) (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- ingest_date: date (nullable = true)
 |-- coupon_code: string (nullable = true)
 |-- ingest_ts: timestamp (nullable = true)
 |-- load_type: string (nullable = true)
 |-- source_file: str

In [0]:
# PREPARE GOLD ORDERS

orders_gold = (
    spark.table(
        f"{CATALOG}.{SILVER_SCHEMA}.silver1_orders_clean"
    )
    .select(
        "order_id",
        "order_ts",
        "customer_id",
        "product_id",
        "store_id",
        "quantity",
        "unit_price",
        "discount_pct",
        "gross_amount",
        "payment_method",
        "order_status",
        "ingest_date",
        "coupon_code",
        "ingest_ts",
        "load_type",
        "source_file"
    )
    .withColumn(
        "order_date",
        F.to_date(F.col("order_ts"))
    )
)
print("Orders:", orders_gold.count())

print(
    "Distinct order IDs:",
    orders_gold.select("order_id").distinct().count()
)

print(
    "Null order IDs:",
    orders_gold.filter(F.col("order_id").isNull()).count()
)

print(
    "Null order dates:",
    orders_gold.filter(F.col("order_date").isNull()).count()
)

orders_gold.printSchema()

display(orders_gold.limit(10))

Orders: 5716
Distinct order IDs: 5716
Null order IDs: 0
Null order dates: 0
root
 |-- order_id: string (nullable = true)
 |-- order_ts: timestamp (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: decimal(18,2) (nullable = true)
 |-- discount_pct: double (nullable = true)
 |-- gross_amount: decimal(18,2) (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- ingest_date: date (nullable = true)
 |-- coupon_code: string (nullable = true)
 |-- ingest_ts: timestamp (nullable = true)
 |-- load_type: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- order_date: date (nullable = true)



order_id,order_ts,customer_id,product_id,store_id,quantity,unit_price,discount_pct,gross_amount,payment_method,order_status,ingest_date,coupon_code,ingest_ts,load_type,source_file,order_date
OI1000001,2026-04-24T06:19:00.000Z,C01655,P00595,S003,6,6139.00,0.2,36834.00,WALLET,shipped,2026-04-24,null,2026-08-11T17:49:51.127Z,incremental,/Volumes/retail_demo/raw/retail_files/retail_delta_project/datasets/incremental/day_2026-04-24/orders_incremental_2026-04-24.csv,2026-04-24
OI1000002,2026-04-24T18:32:00.000Z,C01699,P00359,S026,4,74405.37,0.2,282740.41,NETBANKING,delivered,2026-04-24,null,2026-08-11T17:49:51.127Z,incremental,/Volumes/retail_demo/raw/retail_files/retail_delta_project/datasets/incremental/day_2026-04-24/orders_incremental_2026-04-24.csv,2026-04-24
OI1000003,2026-04-24T22:28:00.000Z,C02173,P00026,S043,4,61875.00,0.05,198000.00,WALLET,returned,2026-04-24,null,2026-08-11T17:49:51.127Z,incremental,/Volumes/retail_demo/raw/retail_files/retail_delta_project/datasets/incremental/day_2026-04-24/orders_incremental_2026-04-24.csv,2026-04-24
OI1000004,2026-04-24T10:28:00.000Z,C00269,P00227,S017,1,57830.24,0.0,46264.19,UPI,delivered,2026-04-24,null,2026-08-11T17:49:51.127Z,incremental,/Volumes/retail_demo/raw/retail_files/retail_delta_project/datasets/incremental/day_2026-04-24/orders_incremental_2026-04-24.csv,2026-04-24
OI1000006,2026-04-24T04:59:00.000Z,C00910,P00322,S038,2,64831.95,0.0,129663.90,UPI,delivered,2026-04-24,null,2026-08-11T17:49:51.127Z,incremental,/Volumes/retail_demo/raw/retail_files/retail_delta_project/datasets/incremental/day_2026-04-24/orders_incremental_2026-04-24.csv,2026-04-24
OI1000007,2026-04-24T15:52:00.000Z,C00332,P00088,S009,1,65627.82,0.05,52502.26,COD,returned,2026-04-24,null,2026-08-11T17:49:51.127Z,incremental,/Volumes/retail_demo/raw/retail_files/retail_delta_project/datasets/incremental/day_2026-04-24/orders_incremental_2026-04-24.csv,2026-04-24
OI1000008,2026-04-24T21:20:00.000Z,C00420,P00305,S010,3,18907.54,0.2,45378.10,CARD,cancelled,2026-04-24,null,2026-08-11T17:49:51.127Z,incremental,/Volumes/retail_demo/raw/retail_files/retail_delta_project/datasets/incremental/day_2026-04-24/orders_incremental_2026-04-24.csv,2026-04-24
OI1000009,2026-04-24T05:26:00.000Z,C01581,P00354,S044,1,17006.45,0.2,17006.45,UPI,returned,2026-04-24,null,2026-08-11T17:49:51.127Z,incremental,/Volumes/retail_demo/raw/retail_files/retail_delta_project/datasets/incremental/day_2026-04-24/orders_incremental_2026-04-24.csv,2026-04-24
OI1000010,2026-04-24T02:09:00.000Z,C02194,P00170,S010,1,72347.53,0.1,68730.15,UPI,cancelled,2026-04-24,null,2026-08-11T17:49:51.127Z,incremental,/Volumes/retail_demo/raw/retail_files/retail_delta_project/datasets/incremental/day_2026-04-24/orders_incremental_2026-04-24.csv,2026-04-24
OI1000011,2026-04-24T11:58:00.000Z,C00038,P00632,S016,4,62788.56,0.2,226038.82,CARD,shipped,2026-04-24,null,2026-08-11T17:49:51.127Z,incremental,/Volumes/retail_demo/raw/retail_files/retail_delta_project/datasets/incremental/day_2026-04-24/orders_incremental_2026-04-24.csv,2026-04-24


In [0]:
#  POINT-IN-TIME CUSTOMER SCD2 JOIN

customer_dim = (
    spark.table(
        f"{CATALOG}.{SILVER_SCHEMA}.dim_customer_scd2"
    )
    .select(
        F.col("customer_id").alias("dim_customer_id"),
        F.col("customer_sk"),
        F.col("effective_start_date").alias("customer_start_date"),
        F.col("effective_end_date").alias("customer_end_date"),
        F.col("is_current").alias("customer_is_current"),
        F.col("customer_name"),
        F.col("city").alias("customer_city"),
        F.col("segment").alias("customer_segment"),
        F.col("gender").alias("customer_gender"),
        F.col("status").alias("customer_status")
    )
)

orders_customer = (
    orders_gold.alias("o")
    .join(
        customer_dim.alias("c"),
        (
            (F.col("o.customer_id") == F.col("c.dim_customer_id"))
            &
            (
                F.col("o.order_date") >=
                F.col("c.customer_start_date")
            )
            &
            (
                F.col("o.order_date") <=
                F.col("c.customer_end_date")
            )
        ),
        "left"
    )
    .select(
        F.col("o.*"),
        F.col("c.customer_sk"),
        F.col("c.customer_name"),
        F.col("c.customer_city"),
        F.col("c.customer_segment"),
        F.col("c.customer_gender"),
        F.col("c.customer_status")
    )
)
print("Orders:", orders_customer.count())

print(
    "Orders with customer_sk:",
    orders_customer
    .filter(F.col("customer_sk").isNotNull())
    .count()
)

print(
    "Orders missing customer_sk:",
    orders_customer
    .filter(F.col("customer_sk").isNull())
    .count()
)

display(
    orders_customer.select(
        "order_id",
        "order_date",
        "customer_id",
        "customer_sk",
        "customer_name",
        "customer_city",
        "customer_segment"
    ).limit(10)
)

Orders: 5716
Orders with customer_sk: 5658
Orders missing customer_sk: 58


order_id,order_date,customer_id,customer_sk,customer_name,customer_city,customer_segment
OI1000001,2026-04-24,C01655,e5c7a565c2600de9b5eda9ec105ec46542476f23db59e84b5746cc3cbb3c4ab5,Customer_1655,Chennai,Gold
OI1000002,2026-04-24,C01699,601e9aa14e301470926fb98729f67cd39e2d83d9c2369bbe72e120c8cd5e2baf,Customer_1699,Kolkata,Platinum
OI1000003,2026-04-24,C02173,b6f5697e0f8e7cc4217e263d838874f219ca0dac6fcd333d29f131d078b229b6,Customer_2173,Chennai,Silver
OI1000004,2026-04-24,C00269,e6b35e0cb8aa5046680a276f7719888300d2fb35b99c22639ca6dcaf584511cc,Customer_269,Pune,Gold
OI1000006,2026-04-24,C00910,c536df2707135e0de94b9cca8d3e774849f5dcf86352d4de62ad2503d50d3834,Customer_910,Gurugram,Regular
OI1000007,2026-04-24,C00332,8563b270b2829417f4cc2a8d24714cbbbda8c8ff888307af1d0ea1a982a3c18e,Customer_332,Kolkata,Silver
OI1000008,2026-04-24,C00420,f6967501f9a6c180bbe5334b5b2f6ef06d6377ce04bf29ffc6eaaa6bf3aebf39,Customer_420,Bengaluru,Regular
OI1000009,2026-04-24,C01581,cb51099d848bb4c41845722fcc7f7ae0e4b7aa85a54bb2657bd51b5756588c0b,Customer_1581,Mumbai,Regular
OI1000010,2026-04-24,C02194,e93bf1fc8ef20733591f52b8c95585efd1a45460b4864234cb89954809e25775,Customer_2194,Bengaluru,Platinum
OI1000011,2026-04-24,C00038,da1f527fd92b8ee7ee8c210a5190587723dcd54b8cf1dd32667dd6731e6bc6fe,Customer_38,Bengaluru,Silver


In [0]:
#  DIAGNOSE MISSING CUSTOMER SCD2 MATCHES

missing_customer_orders = (
    orders_customer
    .filter(F.col("customer_sk").isNull())
)
print(
    "Orders missing customer_sk:",
    missing_customer_orders.count()
)

print(
    "Distinct missing customer IDs:",
    missing_customer_orders
    .select("customer_id")
    .distinct()
    .count()
)

print("\nMissing customer IDs and order dates:")

display(
    missing_customer_orders
    .select(
        "order_id",
        "order_date",
        "customer_id"
    )
    .orderBy("customer_id", "order_date")
    .limit(100)
)

customer_dimension_ids = (
    spark.table(
        f"{CATALOG}.{SILVER_SCHEMA}.dim_customer_scd2"
    )
    .select(
        F.col("customer_id").alias("dim_customer_id")
    )
    .distinct()
)

missing_customer_analysis = (
    missing_customer_orders
    .select("customer_id")
    .distinct()
    .join(
        customer_dimension_ids,
        F.col("customer_id") ==
        F.col("dim_customer_id"),
        "left"
    )
    .withColumn(
        "exists_in_customer_dim",
        F.col("dim_customer_id").isNotNull()
    )
)

print("\nCustomer dimension existence check:")

display(
    missing_customer_analysis
    .groupBy("exists_in_customer_dim")
    .count()
)

Orders missing customer_sk: 58
Distinct missing customer IDs: 23

Missing customer IDs and order dates:


order_id,order_date,customer_id
OI1000249,2026-04-24,C00035
OI1000514,2026-04-24,C00035
OI2000774,2026-04-25,C00035
OI3001033,2026-04-26,C00035
OI3000523,2026-04-20,C00066
OI3000015,2026-04-26,C00066
OI1001201,2026-04-24,C00350
OI2000911,2026-04-25,C00350
OI1001161,2026-04-19,C00410
OI2000779,2026-04-25,C00410



Customer dimension existence check:


exists_in_customer_dim,count
false,23


In [0]:
#  CUSTOMER SCD2 DATE-GAP ANALYSIS

missing_customer_ids = (
    missing_customer_orders
    .select("customer_id")
    .distinct()
)
customer_history = (
    spark.table(
        f"{CATALOG}.{SILVER_SCHEMA}.dim_customer_scd2"
    )
    .select(
        "customer_id",
        "effective_start_date",
        "effective_end_date",
        "is_current",
        "customer_sk"
    )
)
customer_gap_analysis = (
    missing_customer_orders.alias("o")
    .join(
        customer_history.alias("d"),
        F.col("o.customer_id") == F.col("d.customer_id"),
        "left"
    )
    .groupBy(
        F.col("o.customer_id").alias("customer_id")
    )
    .agg(
        F.count("*").alias("missing_order_rows"),
        F.min(F.col("o.order_date")).alias("first_order_date"),
        F.max(F.col("o.order_date")).alias("last_order_date"),
        F.min(F.col("d.effective_start_date")).alias("dimension_start"),
        F.max(F.col("d.effective_end_date")).alias("dimension_end")
    )
    .orderBy("customer_id")
)
display(customer_gap_analysis)
print("\nSummary:")
display(
    customer_gap_analysis
    .withColumn(
        "dimension_exists",
        F.col("dimension_start").isNotNull()
    )
    .groupBy("dimension_exists")
    .agg(
        F.count("*").alias("customer_ids"),
        F.sum("missing_order_rows").alias("orders")
    )
)

customer_id,missing_order_rows,first_order_date,last_order_date,dimension_start,dimension_end
C00035,4,2026-04-24,2026-04-26,null,null
C00066,2,2026-04-20,2026-04-26,null,null
C00350,2,2026-04-24,2026-04-25,null,null
C00410,2,2026-04-19,2026-04-25,null,null
C00473,1,2026-04-25,2026-04-25,null,null
C00710,1,2026-04-24,2026-04-24,null,null
C00801,1,2026-04-24,2026-04-24,null,null
C00857,3,2026-04-24,2026-04-26,null,null
C00956,2,2026-04-25,2026-04-25,null,null
C01006,5,2026-04-26,2026-04-26,null,null



Summary:


dimension_exists,customer_ids,orders
false,23,58


In [0]:
# TRACE MISSING CUSTOMER IDS

missing_ids = (
    missing_customer_orders
    .select("customer_id")
    .distinct()
)
silver_customers_source = (
    spark.table(
        f"{CATALOG}.{SILVER_SCHEMA}.silver1_customers_clean"
    )
    .select(
        "customer_id",
        "signup_date",
        "effective_date",
        "operation",
        "ingest_ts"
    )
)

customer_cdc_source = (
    spark.table(
        f"{CATALOG}.{RAW_SCHEMA}.bronze_customers_incremental"
    )
    .select(
        "customer_id",
        "signup_date",
        "effective_date",
        "operation",
        "ingest_ts"
    )
)

missing_customer_trace = (
    missing_ids
    .join(
        silver_customers_source,
        on="customer_id",
        how="left"
    )
    .groupBy("customer_id")
    .agg(
        F.count("operation").alias("silver_rows"),
        F.min("effective_date").alias("silver_min_effective_date"),
        F.max("effective_date").alias("silver_max_effective_date")
    )
)
cdc_trace = (
    missing_ids
    .join(
        customer_cdc_source,
        on="customer_id",
        how="left"
    )
    .groupBy("customer_id")
    .agg(
        F.count("operation").alias("cdc_rows"),
        F.collect_set("operation").alias("cdc_operations"),
        F.min("effective_date").alias("cdc_min_effective_date"),
        F.max("effective_date").alias("cdc_max_effective_date")
    )
)

final_trace = (
    missing_customer_trace
    .join(
        cdc_trace,
        on="customer_id",
        how="outer"
    )
    .orderBy("customer_id")
)
display(final_trace)
print("\nSUMMARY:")
display(
    final_trace
    .withColumn(
        "in_silver",
        F.col("silver_rows") > 0
    )
    .withColumn(
        "in_cdc",
        F.col("cdc_rows") > 0
    )
    .groupBy("in_silver", "in_cdc")
    .agg(
        F.count("*").alias("customer_ids")
    )
    .orderBy("in_silver", "in_cdc")
)

customer_id,silver_rows,silver_min_effective_date,silver_max_effective_date,cdc_rows,cdc_operations,cdc_min_effective_date,cdc_max_effective_date
C00035,0,null,null,0,List(),null,null
C00066,0,null,null,1,List(UPDATE),2026-04-24,2026-04-24
C00350,0,null,null,0,List(),null,null
C00410,0,null,null,0,List(),null,null
C00473,0,null,null,0,List(),null,null
C00710,0,null,null,1,List(UPDATE),2026-04-26,2026-04-26
C00801,0,null,null,0,List(),null,null
C00857,0,null,null,0,List(),null,null
C00956,0,null,null,0,List(),null,null
C01006,0,null,null,0,List(),null,null



SUMMARY:


in_silver,in_cdc,customer_ids
false,false,20
false,true,3


In [0]:
# CUSTOMER SCD2 MATCH VALIDATION

matched_customer_orders = (
    orders_customer
    .filter(F.col("customer_sk").isNotNull())
)
unmatched_customer_orders = (
    orders_customer
    .filter(F.col("customer_sk").isNull())
)
print("Total orders:", orders_customer.count())
print(
    "Orders with customer_sk:",
    matched_customer_orders.count()
)
print(
    "Orders without customer_sk:",
    unmatched_customer_orders.count()
)
print(
    "Distinct unmatched customer IDs:",
    unmatched_customer_orders
    .select("customer_id")
    .distinct()
    .count()
)
print("\nUnmatched Customer IDs:")
display(
    unmatched_customer_orders
    .select(
        "customer_id",
        "order_date"
    )
    .distinct()
    .orderBy("customer_id", "order_date")
)

print("\nMatch summary:")

display(
    orders_customer
    .withColumn(
        "customer_match_status",
        F.when(
            F.col("customer_sk").isNotNull(),
            F.lit("MATCHED")
        ).otherwise(
            F.lit("UNMATCHED")
        )
    )
    .groupBy("customer_match_status")
    .count()
)

Total orders: 5716
Orders with customer_sk: 5658
Orders without customer_sk: 525
Distinct unmatched customer IDs: 263

Unmatched Customer IDs:


customer_id,order_date
C00015,2026-04-25
C00020,2026-04-26
C00022,2026-04-26
C00035,2026-04-24
C00035,2026-04-25
C00035,2026-04-26
C00039,2026-04-25
C00044,2026-04-24
C00044,2026-04-25
C00044,2026-04-26



Match summary:


customer_match_status,count
UNMATCHED,58
MATCHED,5658


In [0]:
#  POINT-IN-TIME PRODUCT SCD2 JOIN


product_dim = (
    spark.table(
        f"{CATALOG}.{SILVER_SCHEMA}.dim_product_scd2"
    )
    .select(
        F.col("product_id").alias("dim_product_id"),
        F.col("product_sk"),
        F.col("effective_start_date").alias("product_start_date"),
        F.col("effective_end_date").alias("product_end_date"),
        F.col("is_current").alias("product_is_current"),
        F.col("product_name"),
        F.col("category").alias("product_category"),
        F.col("brand").alias("product_brand"),
        F.col("unit_price").alias("product_unit_price"),
        F.col("status").alias("product_status")
    )
)

orders_customer_product = (
    orders_customer.alias("o")
    .join(
        product_dim.alias("p"),
        (
            (F.col("o.product_id") == F.col("p.dim_product_id"))
            &
            (
                F.col("o.order_date") >=
                F.col("p.product_start_date")
            )
            &
            (
                F.col("o.order_date") <=
                F.col("p.product_end_date")
            )
        ),
        "left"
    )
    .select(
        F.col("o.*"),
        F.col("p.product_sk"),
        F.col("p.product_name"),
        F.col("p.product_category"),
        F.col("p.product_brand"),
        F.col("p.product_unit_price"),
        F.col("p.product_status")
    )
)
print("Orders:", orders_customer_product.count())

print(
    "Orders with product_sk:",
    orders_customer_product
    .filter(F.col("product_sk").isNotNull())
    .count()
)

print(
    "Orders missing product_sk:",
    orders_customer_product
    .filter(F.col("product_sk").isNull())
    .count()
)

print(
    "Distinct unmatched product IDs:",
    orders_customer_product
    .filter(F.col("product_sk").isNull())
    .select("product_id")
    .distinct()
    .count()
)

display(
    orders_customer_product.select(
        "order_id",
        "order_date",
        "product_id",
        "product_sk",
        "product_name",
        "product_category",
        "product_brand"
    ).limit(10)
)

Orders: 5716
Orders with product_sk: 5716
Orders missing product_sk: 0
Distinct unmatched product IDs: 0


order_id,order_date,product_id,product_sk,product_name,product_category,product_brand
OI1000001,2026-04-24,P00595,7ad91181879feade06984dd1cef50e352ca968fb267cf28080b0614b7b0e2dbb,T-Shirt 595,Fashion,BrandA
OI1000002,2026-04-24,P00359,0f2ebe35fecc192cdca2213ededa60de834398d6c7e6373a61f320de10db505c,Jacket 359,Fashion,BrandC
OI1000003,2026-04-24,P00026,bc055b15758099a756069ebd2ef6612674a7d9b3442769699d77025b62970bc9,Tea 26,Grocery,BrandA
OI1000004,2026-04-24,P00227,f2949510aa2b016266c233f9b878dc834bcafb22bb386147078e8ab759b49db8,T-Shirt 227,Fashion,BrandD
OI1000008,2026-04-24,P00305,bf46c70627b2ce28765db4135516977443a45f44ea2aeb0fd4bdebeef825f2fe,Chair 305,Home,BrandC
OI1000009,2026-04-24,P00354,b7455a856018fed19a63362af11ee494bb89d790bd2e4d61acf29e7cb1fe0746,Bedsheet 354,Home,BrandA
OI1000011,2026-04-24,P00632,121bc237e25d024fd641c1845ea5d21c14acff5a48e006e34be90220010860b0,T-Shirt 632,Fashion,BrandC
OI1000006,2026-04-24,P00322,6aaaef29358d888a13fc8d5f821bdfd1b6978cb6b0eb819c396b0ac50752b31f,Shampoo 322,Beauty,BrandB
OI1000007,2026-04-24,P00088,f46387ea8bb9e5a335e0c8f0777b7bbf52c40f297d60a3a88b15e81d95af5c39,Shampoo 88,Beauty,BrandA
OI1000010,2026-04-24,P00170,67e63e2dc85815cac5e68d61c9e9f5d7391d6bfa25d255f4babdd8ed5ca05e9a,Coffee 170,Grocery,BrandC


In [0]:
# STORE DIMENSION JOIN

store_dim_gold = (
    spark.table(
        f"{CATALOG}.{SILVER_SCHEMA}.dim_store"
    )
    .select(
        F.col("store_id").alias("dim_store_id"),
        F.col("store_name"),
        F.col("city").alias("store_city"),
        F.col("region").alias("store_region"),
        F.col("status").alias("store_status")
    )
)

orders_conformed = (
    orders_customer_product.alias("o")
    .join(
        store_dim_gold.alias("s"),
        F.col("o.store_id") == F.col("s.dim_store_id"),
        "left"
    )
    .select(
        F.col("o.*"),
        F.col("s.store_name"),
        F.col("s.store_city"),
        F.col("s.store_region"),
        F.col("s.store_status")
    )
)
print("Orders:", orders_conformed.count())

print(
    "Orders with store:",
    orders_conformed
    .filter(F.col("store_name").isNotNull())
    .count()
)

print(
    "Orders missing store:",
    orders_conformed
    .filter(F.col("store_name").isNull())
    .count()
)

print(
    "Distinct unmatched store IDs:",
    orders_conformed
    .filter(F.col("store_name").isNull())
    .select("store_id")
    .distinct()
    .count()
)

display(
    orders_conformed.select(
        "order_id",
        "order_date",
        "store_id",
        "store_name",
        "store_city",
        "store_region"
    ).limit(10)
)

Orders: 5716
Orders with store: 5716
Orders missing store: 0
Distinct unmatched store IDs: 0


order_id,order_date,store_id,store_name,store_city,store_region
OI1000001,2026-04-24,S003,Store_3,Ahmedabad,South
OI1000002,2026-04-24,S026,Store_26,Ahmedabad,West
OI1000003,2026-04-24,S043,Store_43,Delhi,West
OI1000004,2026-04-24,S017,Store_17,Jaipur,Online
OI1000008,2026-04-24,S010,Store_10,Gurugram,East
OI1000009,2026-04-24,S044,Store_44,Kolkata,West
OI1000011,2026-04-24,S016,Store_16,Ahmedabad,West
OI1000006,2026-04-24,S038,Store_38,Pune,South
OI1000007,2026-04-24,S009,Store_9,Bengaluru,West
OI1000010,2026-04-24,S010,Store_10,Gurugram,East


# GOLD LAYER

In [0]:
# BUILD CONFORMED FACT DATASET

fact_orders = (
    orders_conformed
    .select(
        "order_id",
        "order_ts",
        "order_date",
        "customer_id",
        "customer_sk",
        "customer_name",
        "customer_city",
        "customer_segment",
        "customer_gender",
        "customer_status",
        "product_id",
        "product_sk",
        "product_name",
        "product_category",
        "product_brand",
        "product_unit_price",
        "product_status",
        "store_id",
        "store_name",
        "store_city",
        "store_region",
        "store_status",
        "quantity",
        "unit_price",
        "discount_pct",
        "gross_amount",
        "payment_method",
        "order_status",
        "coupon_code",
        "ingest_date",
        "ingest_ts",
        "load_type",
        "source_file"
    )
)
total_rows = fact_orders.count()
distinct_orders = (
    fact_orders
    .select("order_id")
    .distinct()
    .count()
)
null_order_ids = (
    fact_orders
    .filter(F.col("order_id").isNull())
    .count()
)
null_customer_sk = (
    fact_orders
    .filter(F.col("customer_sk").isNull())
    .count()
)
null_product_sk = (
    fact_orders
    .filter(F.col("product_sk").isNull())
    .count()
)
null_store_ids = (
    fact_orders
    .filter(F.col("store_id").isNull())
    .count()
)

duplicate_orders = (
    fact_orders
    .groupBy("order_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Total fact rows:", total_rows)
print("Distinct order IDs:", distinct_orders)
print("Null order IDs:", null_order_ids)
print("Null customer_sk:", null_customer_sk)
print("Null product_sk:", null_product_sk)
print("Null store IDs:", null_store_ids)
print("Duplicate order IDs:", duplicate_orders)

print("\nFact schema:")
fact_orders.printSchema()

print("\nSample fact records:")
display(
    fact_orders.limit(10)
)

Total fact rows: 5716
Distinct order IDs: 5716
Null order IDs: 0
Null customer_sk: 58
Null product_sk: 0
Null store IDs: 0
Duplicate order IDs: 0

Fact schema:
root
 |-- order_id: string (nullable = true)
 |-- order_ts: timestamp (nullable = true)
 |-- order_date: date (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_sk: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_segment: string (nullable = true)
 |-- customer_gender: string (nullable = true)
 |-- customer_status: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_sk: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- product_brand: string (nullable = true)
 |-- product_unit_price: decimal(18,2) (nullable = true)
 |-- product_status: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- store_name: string (nulla

order_id,order_ts,order_date,customer_id,customer_sk,customer_name,customer_city,customer_segment,customer_gender,customer_status,product_id,product_sk,product_name,product_category,product_brand,product_unit_price,product_status,store_id,store_name,store_city,store_region,store_status,quantity,unit_price,discount_pct,gross_amount,payment_method,order_status,coupon_code,ingest_date,ingest_ts,load_type,source_file
OI1000001,2026-04-24T06:19:00.000Z,2026-04-24,C01655,e5c7a565c2600de9b5eda9ec105ec46542476f23db59e84b5746cc3cbb3c4ab5,Customer_1655,Chennai,Gold,F,active,P00595,7ad91181879feade06984dd1cef50e352ca968fb267cf28080b0614b7b0e2dbb,T-Shirt 595,Fashion,BrandA,8770.26,active,S003,Store_3,Ahmedabad,South,closed,6,6139.00,0.2,36834.00,WALLET,shipped,null,2026-04-24,2026-08-11T17:49:51.127Z,incremental,/Volumes/retail_demo/raw/retail_files/retail_delta_project/datasets/incremental/day_2026-04-24/orders_incremental_2026-04-24.csv
OI1000002,2026-04-24T18:32:00.000Z,2026-04-24,C01699,601e9aa14e301470926fb98729f67cd39e2d83d9c2369bbe72e120c8cd5e2baf,Customer_1699,Kolkata,Platinum,M,active,P00359,0f2ebe35fecc192cdca2213ededa60de834398d6c7e6373a61f320de10db505c,Jacket 359,Fashion,BrandC,74405.37,discontinued,S026,Store_26,Ahmedabad,West,closed,4,74405.37,0.2,282740.41,NETBANKING,delivered,null,2026-04-24,2026-08-11T17:49:51.127Z,incremental,/Volumes/retail_demo/raw/retail_files/retail_delta_project/datasets/incremental/day_2026-04-24/orders_incremental_2026-04-24.csv
OI1000003,2026-04-24T22:28:00.000Z,2026-04-24,C02173,b6f5697e0f8e7cc4217e263d838874f219ca0dac6fcd333d29f131d078b229b6,Customer_2173,Chennai,Silver,F,active,P00026,bc055b15758099a756069ebd2ef6612674a7d9b3442769699d77025b62970bc9,Tea 26,Grocery,BrandA,61875.00,active,S043,Store_43,Delhi,West,closed,4,61875.00,0.05,198000.00,WALLET,returned,null,2026-04-24,2026-08-11T17:49:51.127Z,incremental,/Volumes/retail_demo/raw/retail_files/retail_delta_project/datasets/incremental/day_2026-04-24/orders_incremental_2026-04-24.csv
OI1000004,2026-04-24T10:28:00.000Z,2026-04-24,C00269,e6b35e0cb8aa5046680a276f7719888300d2fb35b99c22639ca6dcaf584511cc,Customer_269,Pune,Gold,M,active,P00227,f2949510aa2b016266c233f9b878dc834bcafb22bb386147078e8ab759b49db8,T-Shirt 227,Fashion,BrandD,57830.24,discontinued,S017,Store_17,Jaipur,Online,active,1,57830.24,0.0,46264.19,UPI,delivered,null,2026-04-24,2026-08-11T17:49:51.127Z,incremental,/Volumes/retail_demo/raw/retail_files/retail_delta_project/datasets/incremental/day_2026-04-24/orders_incremental_2026-04-24.csv
OI1000008,2026-04-24T21:20:00.000Z,2026-04-24,C00420,f6967501f9a6c180bbe5334b5b2f6ef06d6377ce04bf29ffc6eaaa6bf3aebf39,Customer_420,Bengaluru,Regular,F,inactive,P00305,bf46c70627b2ce28765db4135516977443a45f44ea2aeb0fd4bdebeef825f2fe,Chair 305,Home,BrandC,65024.97,discontinued,S010,Store_10,Gurugram,East,active,3,18907.54,0.2,45378.10,CARD,cancelled,null,2026-04-24,2026-08-11T17:49:51.127Z,incremental,/Volumes/retail_demo/raw/retail_files/retail_delta_project/datasets/incremental/day_2026-04-24/orders_incremental_2026-04-24.csv
OI1000009,2026-04-24T05:26:00.000Z,2026-04-24,C01581,cb51099d848bb4c41845722fcc7f7ae0e4b7aa85a54bb2657bd51b5756588c0b,Customer_1581,Mumbai,Regular,F,active,P00354,b7455a856018fed19a63362af11ee494bb89d790bd2e4d61acf29e7cb1fe0746,Bedsheet 354,Home,BrandA,17006.45,discontinued,S044,Store_44,Kolkata,West,closed,1,17006.45,0.2,17006.45,UPI,returned,null,2026-04-24,2026-08-11T17:49:51.127Z,incremental,/Volumes/retail_demo/raw/retail_files/retail_delta_project/datasets/incremental/day_2026-04-24/orders_incremental_2026-04-24.csv
OI1000011,2026-04-24T11:58:00.000Z,2026-04-24,C00038,da1f527fd92b8ee7ee8c210a5190587723dcd54b8cf1dd32667dd6731e6bc6fe,Customer_38,Bengaluru,Silver,M,active,P00632,121bc237e25d024fd641c1845ea5d21c14acff5a48e006e34be90220010860b0,T-Shirt 632,Fashion,BrandC,62788.56,discontinued,S016,Store_16,Ahmedabad,West,active,4,62788.56,0.2,226038.82,CARD,shipped,null,2026-04-24,2026-08-11T17:49:51.127Z,incremental,/Volu

In [0]:
#  FINAL FACT VALIDATION

print("=" * 70)
print("FINAL GOLD FACT VALIDATION")
print("=" * 70)


fact_row_count = fact_orders.count()

distinct_order_count = (
    fact_orders
    .select("order_id")
    .distinct()
    .count()
)

duplicate_order_ids = (
    fact_orders
    .groupBy("order_id")
    .count()
    .filter(F.col("count") > 1)
)

duplicate_order_count = duplicate_order_ids.count()
null_order_id_count = (
    fact_orders
    .filter(F.col("order_id").isNull())
    .count()
)
null_product_sk_count = (
    fact_orders
    .filter(F.col("product_sk").isNull())
    .count()
)
null_store_id_count = (
    fact_orders
    .filter(F.col("store_id").isNull())
    .count()
)
null_customer_sk_count = (
    fact_orders
    .filter(F.col("customer_sk").isNull())
    .count()
)
null_quantity_count = (
    fact_orders
    .filter(F.col("quantity").isNull())
    .count()
)
null_gross_amount_count = (
    fact_orders
    .filter(F.col("gross_amount").isNull())
    .count()
)
print("Fact rows:", fact_row_count)
print("Distinct orders:", distinct_order_count)
print("Duplicate order IDs:", duplicate_order_count)

print("\nCritical key checks:")
print("Null order IDs:", null_order_id_count)
print("Null customer_sk:", null_customer_sk_count)
print("Null product_sk:", null_product_sk_count)
print("Null store IDs:", null_store_id_count)

print("\nMeasure checks:")
print("Null quantity:", null_quantity_count)
print("Null gross_amount:", null_gross_amount_count)

print("\nFinal validation status:")

if (
    fact_row_count == distinct_order_count
    and duplicate_order_count == 0
    and null_order_id_count == 0
    and null_product_sk_count == 0
    and null_store_id_count == 0
):
    print(" FACT GRAIN VALID")
else:
    print(" FACT GRAIN INVALID")

print("\nSample:")
display(
    fact_orders.select(
        "order_id",
        "order_date",
        "customer_id",
        "customer_sk",
        "product_id",
        "product_sk",
        "store_id",
        "quantity",
        "gross_amount"
    ).limit(10)
)

FINAL GOLD FACT VALIDATION
Fact rows: 5716
Distinct orders: 5716
Duplicate order IDs: 0

Critical key checks:
Null order IDs: 0
Null customer_sk: 57
Null product_sk: 0
Null store IDs: 0

Measure checks:
Null quantity: 0
Null gross_amount: 0

Final validation status:
 FACT GRAIN VALID

Sample:


order_id,order_date,customer_id,customer_sk,product_id,product_sk,store_id,quantity,gross_amount
OI1000001,2026-04-24,C01655,e5c7a565c2600de9b5eda9ec105ec46542476f23db59e84b5746cc3cbb3c4ab5,P00595,7ad91181879feade06984dd1cef50e352ca968fb267cf28080b0614b7b0e2dbb,S003,6,36834.00
OI1000002,2026-04-24,C01699,601e9aa14e301470926fb98729f67cd39e2d83d9c2369bbe72e120c8cd5e2baf,P00359,0f2ebe35fecc192cdca2213ededa60de834398d6c7e6373a61f320de10db505c,S026,4,282740.41
OI1000003,2026-04-24,C02173,b6f5697e0f8e7cc4217e263d838874f219ca0dac6fcd333d29f131d078b229b6,P00026,bc055b15758099a756069ebd2ef6612674a7d9b3442769699d77025b62970bc9,S043,4,198000.00
OI1000004,2026-04-24,C00269,e6b35e0cb8aa5046680a276f7719888300d2fb35b99c22639ca6dcaf584511cc,P00227,f2949510aa2b016266c233f9b878dc834bcafb22bb386147078e8ab759b49db8,S017,1,46264.19
OI1000008,2026-04-24,C00420,f6967501f9a6c180bbe5334b5b2f6ef06d6377ce04bf29ffc6eaaa6bf3aebf39,P00305,bf46c70627b2ce28765db4135516977443a45f44ea2aeb0fd4bdebeef825f2fe,S010,3,45378.10
OI1000009,2026-04-24,C01581,cb51099d848bb4c41845722fcc7f7ae0e4b7aa85a54bb2657bd51b5756588c0b,P00354,b7455a856018fed19a63362af11ee494bb89d790bd2e4d61acf29e7cb1fe0746,S044,1,17006.45
OI1000011,2026-04-24,C00038,da1f527fd92b8ee7ee8c210a5190587723dcd54b8cf1dd32667dd6731e6bc6fe,P00632,121bc237e25d024fd641c1845ea5d21c14acff5a48e006e34be90220010860b0,S016,4,226038.82
OI1000006,2026-04-24,C00910,c536df2707135e0de94b9cca8d3e774849f5dcf86352d4de62ad2503d50d3834,P00322,6aaaef29358d888a13fc8d5f821bdfd1b6978cb6b0eb819c396b0ac50752b31f,S038,2,129663.90
OI1000010,2026-04-24,C02194,e93bf1fc8ef20733591f52b8c95585efd1a45460b4864234cb89954809e25775,P00170,67e63e2dc85815cac5e68d61c9e9f5d7391d6bfa25d255f4babdd8ed5ca05e9a,S010,1,68730.15
OI1000007,2026-04-24,C00332,b9dfb073e5997587eb3aec164a10845e211109d930369e6d09a4b4ea3576cd76,P00088,f46387ea8bb9e5a335e0c8f0777b7bbf52c40f297d60a3a88b15e81d95af5c39,S009,1,52502.26


In [0]:
# WRITE GOLD FACT ORDERS
FACT_ORDERS_TABLE = (
    f"{CATALOG}.{GOLD_SCHEMA}.fact_orders"
)

(
    fact_orders
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(FACT_ORDERS_TABLE)
)
print("Table:", FACT_ORDERS_TABLE)

display(
    spark.sql(f"""
        SELECT
            COUNT(*) AS total_rows,
            COUNT(DISTINCT order_id) AS distinct_orders,
            SUM(quantity) AS total_units,
            SUM(gross_amount) AS total_revenue
        FROM {FACT_ORDERS_TABLE}
    """)
)

Table: retail_demo.gold.fact_orders


total_rows,distinct_orders,total_units,total_revenue
5716,5716,19864,803347391.80


In [0]:
#  GOLD DAILY SALES

daily_sales = (
    fact_orders
    .groupBy("order_date")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.sum("gross_amount").alias("total_revenue"),
        F.sum("quantity").alias("total_units"),
        (
            F.sum("gross_amount") /
            F.countDistinct("order_id")
        ).alias("average_order_value")
    )
    .orderBy("order_date")
)
print("Days:", daily_sales.count())
daily_sales.printSchema()
display(daily_sales.limit(20))

Days: 24
root
 |-- order_date: date (nullable = true)
 |-- total_orders: long (nullable = false)
 |-- total_revenue: decimal(28,2) (nullable = true)
 |-- total_units: long (nullable = true)
 |-- average_order_value: decimal(38,12) (nullable = true)



order_date,total_orders,total_revenue,total_units,average_order_value
2026-04-03,4,545056.01,12,136264.002500000000
2026-04-04,8,1163750.41,27,145468.801250000000
2026-04-05,13,2523024.95,51,194078.842307692308
2026-04-06,13,2182961.34,54,167920.103076923077
2026-04-07,18,2622922.61,60,145717.922777777778
2026-04-08,22,3148971.01,81,143135.045909090909
2026-04-09,19,2845354.48,62,149755.498947368421
2026-04-10,12,1538614.64,42,128217.886666666667
2026-04-11,14,2156540.88,57,154038.634285714286
2026-04-12,17,2314917.47,54,136171.615882352941


In [0]:
#  WRITE GOLD DAILY SALES

DAILY_SALES_TABLE = (
    f"{CATALOG}.{GOLD_SCHEMA}.gold_daily_sales"
)
(
    daily_sales
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(DAILY_SALES_TABLE)
)
print("Table:", DAILY_SALES_TABLE)
display(
    spark.sql(f"""
        SELECT
            COUNT(*) AS days,
            SUM(total_orders) AS total_orders,
            SUM(total_revenue) AS total_revenue,
            SUM(total_units) AS total_units
        FROM {DAILY_SALES_TABLE}
    """)
)

Table: retail_demo.gold.gold_daily_sales


days,total_orders,total_revenue,total_units
24,5716,803347391.80,19864


In [0]:
#  GOLD CATEGORY SALES

from pyspark.sql import functions as F
FACT_ORDERS_TABLE = "retail_demo.gold.fact_orders"
fact_orders_gold = spark.table(
    FACT_ORDERS_TABLE
)
category_sales = (
    fact_orders_gold
    .groupBy("product_category")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.sum("gross_amount").alias("total_revenue"),
        F.sum("quantity").alias("total_units")
    )
    .orderBy(F.col("total_revenue").desc())
)
print("Categories:", category_sales.count())
category_sales.printSchema()
display(category_sales)

Categories: 7
root
 |-- product_category: string (nullable = true)
 |-- total_orders: long (nullable = false)
 |-- total_revenue: decimal(28,2) (nullable = true)
 |-- total_units: long (nullable = true)



product_category,total_orders,total_revenue,total_units
Home,1321,175889818.82,4624
Fashion,1218,170703562.45,4205
Grocery,1045,158045921.79,3620
Electronics,1040,141623368.40,3642
Beauty,984,141063023.45,3429
Unknown,96,13854391.62,301
null,12,2167305.27,43


In [0]:
#  WRITE GOLD CATEGORY SALES

CATEGORY_SALES_TABLE = "retail_demo.gold.gold_category_sales"

(
    category_sales
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(CATEGORY_SALES_TABLE)
)
print("Table:", CATEGORY_SALES_TABLE)

display(
    spark.sql(f"""
        SELECT
            COUNT(*) AS categories,
            SUM(total_orders) AS total_orders,
            SUM(total_revenue) AS total_revenue,
            SUM(total_units) AS total_units
        FROM {CATEGORY_SALES_TABLE}
    """)
)

Table: retail_demo.gold.gold_category_sales


categories,total_orders,total_revenue,total_units
7,5716,803347391.80,19864


In [0]:
# GOLD SEGMENT SALES

from pyspark.sql import functions as F

FACT_ORDERS_TABLE = "retail_demo.gold.fact_orders"

fact_orders_gold = spark.table(
    FACT_ORDERS_TABLE
)

segment_sales = (
    fact_orders_gold
    .groupBy("customer_segment")
    .agg(
        F.countDistinct("customer_id").alias("unique_customers"),
        F.countDistinct("order_id").alias("total_orders"),
        F.sum("gross_amount").alias("total_revenue")
    )
    .orderBy(F.col("total_revenue").desc())
)
print(
    "Segments:",
    segment_sales.count()
)
segment_sales.printSchema()
display(segment_sales)

Segments: 5
root
 |-- customer_segment: string (nullable = true)
 |-- unique_customers: long (nullable = false)
 |-- total_orders: long (nullable = false)
 |-- total_revenue: decimal(28,2) (nullable = true)



customer_segment,unique_customers,total_orders,total_revenue
Regular,565,1456,202111031.24
Platinum,560,1382,194837212.58
Gold,529,1333,190649987.93
Silver,549,1322,188375610.70
null,113,223,27373549.35


In [0]:
# WRITE GOLD SEGMENT SALES
SEGMENT_SALES_TABLE = "retail_demo.gold.gold_segment_sales"
(
    segment_sales
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SEGMENT_SALES_TABLE)
)
print("Table:", SEGMENT_SALES_TABLE)
display(
    spark.sql(f"""
        SELECT
            COUNT(*) AS segments,
            SUM(unique_customers) AS customer_count_sum,
            SUM(total_orders) AS total_orders,
            SUM(total_revenue) AS total_revenue
        FROM {SEGMENT_SALES_TABLE}
    """)
)

Table: retail_demo.gold.gold_segment_sales


segments,customer_count_sum,total_orders,total_revenue
5,2316,5716,803347391.80


In [0]:
#GOLD REGION SALES
from pyspark.sql import functions as F
FACT_ORDERS_TABLE = "retail_demo.gold.fact_orders"
fact_orders_gold = spark.table(
    FACT_ORDERS_TABLE
)
region_sales = (
    fact_orders_gold
    .groupBy("store_region")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.sum("gross_amount").alias("total_revenue"),
        F.sum("quantity").alias("total_units")
    )
    .orderBy(F.col("total_revenue").desc())
)
print(
    "Regions:",
    region_sales.count()
)
region_sales.printSchema()
display(region_sales)

Regions: 5
root
 |-- store_region: string (nullable = true)
 |-- total_orders: long (nullable = false)
 |-- total_revenue: decimal(28,2) (nullable = true)
 |-- total_units: long (nullable = true)



store_region,total_orders,total_revenue,total_units
Online,1398,198733595.58,4823
South,1325,189059108.26,4626
North,1122,158536875.73,3928
West,1067,148379720.94,3767
East,804,108638091.29,2720


In [0]:
#  WRITE GOLD REGION SALES
REGION_SALES_TABLE = "retail_demo.gold.gold_region_sales"

(
    region_sales
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(REGION_SALES_TABLE)
)
print("Table:", REGION_SALES_TABLE)
display(
    spark.sql(f"""
        SELECT
            COUNT(*) AS regions,
            SUM(total_orders) AS total_orders,
            SUM(total_revenue) AS total_revenue,
            SUM(total_units) AS total_units
        FROM {REGION_SALES_TABLE}
    """)
)

Table: retail_demo.gold.gold_region_sales


regions,total_orders,total_revenue,total_units
5,5716,803347391.80,19864


In [0]:
#  FINAL GOLD LAYER VALIDATION


gold_tables = {
    "fact_orders": "retail_demo.gold.fact_orders",
    "gold_daily_sales": "retail_demo.gold.gold_daily_sales",
    "gold_category_sales": "retail_demo.gold.gold_category_sales",
    "gold_segment_sales": "retail_demo.gold.gold_segment_sales",
    "gold_region_sales": "retail_demo.gold.gold_region_sales"
}
for table_name, full_name in gold_tables.items():
    try:
        row_count = spark.table(full_name).count()

        print(
            f"✓ {full_name:<50} {row_count:,} rows"
        )
    except Exception as e:
        print(
            f"✗ {full_name:<50} ERROR"
        )
        print(str(e)[:200])

display(
    spark.sql("""
        SELECT
            COUNT(*) AS fact_orders,
            COUNT(DISTINCT order_id) AS distinct_orders,
            SUM(quantity) AS total_units,
            SUM(gross_amount) AS total_revenue
        FROM retail_demo.gold.fact_orders
    """)
)
display(
    spark.sql("""
        SELECT
            SUM(total_orders) AS daily_orders,
            SUM(total_units) AS daily_units,
            SUM(total_revenue) AS daily_revenue
        FROM retail_demo.gold.gold_daily_sales
    """)
)

display(
    spark.sql("""
        SELECT
            SUM(total_orders) AS category_orders,
            SUM(total_units) AS category_units,
            SUM(total_revenue) AS category_revenue
        FROM retail_demo.gold.gold_category_sales
    """)
)

display(
    spark.sql("""
        SELECT
            SUM(total_orders) AS segment_orders,
            SUM(total_revenue) AS segment_revenue
        FROM retail_demo.gold.gold_segment_sales
    """)
)

display(
    spark.sql("""
        SELECT
            SUM(total_orders) AS region_orders,
            SUM(total_units) AS region_units,
            SUM(total_revenue) AS region_revenue
        FROM retail_demo.gold.gold_region_sales
    """)
)

✓ retail_demo.gold.fact_orders                       5,716 rows
✓ retail_demo.gold.gold_daily_sales                  24 rows
✓ retail_demo.gold.gold_category_sales               7 rows
✓ retail_demo.gold.gold_segment_sales                5 rows
✓ retail_demo.gold.gold_region_sales                 5 rows


fact_orders,distinct_orders,total_units,total_revenue
5716,5716,19864,803347391.80


daily_orders,daily_units,daily_revenue
5716,19864,803347391.80


category_orders,category_units,category_revenue
5716,19864,803347391.80


segment_orders,segment_revenue
5716,803347391.80


region_orders,region_units,region_revenue
5716,19864,803347391.80


In [0]:
#  DELTA TABLE HISTORY
fact_history = spark.sql("""
    DESCRIBE HISTORY retail_demo.gold.fact_orders
""")
display(
    fact_history.select(
        "version",
        "timestamp",
        "operation",
        "operationMetrics"
    )
)

version,timestamp,operation,operationMetrics
3,2026-08-13T03:12:09.000Z,CREATE OR REPLACE TABLE AS SELECT,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 298749, numDeletionVectorsRemoved -> 0, numOutputRows -> 5716, numOutputBytes -> 293560)"
2,2026-08-13T02:33:22.000Z,CREATE OR REPLACE TABLE AS SELECT,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 298749, numDeletionVectorsRemoved -> 0, numOutputRows -> 5716, numOutputBytes -> 298749)"
1,2026-08-12T20:16:33.000Z,CREATE OR REPLACE TABLE AS SELECT,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 298749, numDeletionVectorsRemoved -> 0, numOutputRows -> 5716, numOutputBytes -> 298749)"
0,2026-08-12T14:45:48.000Z,CREATE OR REPLACE TABLE AS SELECT,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 5716, numOutputBytes -> 298749)"


In [0]:
#  DELTA TIME TRAVEL

print("=" * 70)
print("DELTA TIME TRAVEL — FACT ORDERS")
print("=" * 70)

fact_version_0 = spark.sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT order_id) AS distinct_orders,
        SUM(quantity) AS total_units,
        SUM(gross_amount) AS total_revenue
    FROM retail_demo.gold.fact_orders VERSION AS OF 0
""")

display(fact_version_0)

DELTA TIME TRAVEL — FACT ORDERS


total_rows,distinct_orders,total_units,total_revenue
5716,5716,19864,803347391.80


In [0]:
#  DELTA SCHEMA EVOLUTION DEMONSTRATION

from pyspark.sql import functions as F

SCHEMA_DEMO_TABLE = (
    "retail_demo.gold.fact_orders_schema_evolution_demo"
)

fact_initial = (
    spark.table(
        "retail_demo.gold.fact_orders"
    )
    .select(
        "order_id",
        "order_ts",
        "order_date",
        "customer_id",
        "customer_sk",
        "product_id",
        "product_sk",
        "store_id",
        "quantity",
        "unit_price",
        "discount_pct",
        "gross_amount",
        "payment_method",
        "order_status"
    )
    .limit(100)
)

(
    fact_initial
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SCHEMA_DEMO_TABLE)
)

print("Initial schema written WITHOUT coupon_code.")

print("\nInitial schema:")

spark.table(
    SCHEMA_DEMO_TABLE
).printSchema()


fact_day3 = (
    spark.table(
        "retail_demo.gold.fact_orders"
    )
    .select(
        "order_id",
        "order_ts",
        "order_date",
        "customer_id",
        "customer_sk",
        "product_id",
        "product_sk",
        "store_id",
        "quantity",
        "unit_price",
        "discount_pct",
        "gross_amount",
        "payment_method",
        "order_status",
        "coupon_code"
    )
    .limit(20)
)

print("\nIncoming schema WITH coupon_code:")

fact_day3.printSchema()

(
    fact_day3
    .write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(SCHEMA_DEMO_TABLE)
)

print("\nSchema evolution append completed.")
evolved_table = spark.table(
    SCHEMA_DEMO_TABLE
)
print("\nFinal evolved schema:")
evolved_table.printSchema()
display(
    spark.sql(f"""
        SELECT
            COUNT(*) AS total_rows,
            COUNT(coupon_code) AS rows_with_coupon_code
        FROM {SCHEMA_DEMO_TABLE}
    """)
)

Initial schema written WITHOUT coupon_code.

Initial schema:
root
 |-- order_id: string (nullable = true)
 |-- order_ts: timestamp (nullable = true)
 |-- order_date: date (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_sk: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_sk: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: decimal(18,2) (nullable = true)
 |-- discount_pct: double (nullable = true)
 |-- gross_amount: decimal(18,2) (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_status: string (nullable = true)


Incoming schema WITH coupon_code:
root
 |-- order_id: string (nullable = true)
 |-- order_ts: timestamp (nullable = true)
 |-- order_date: date (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_sk: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_sk: string (nullable 

total_rows,rows_with_coupon_code
120,0


In [0]:
#  VALIDATE DELTA SCHEMA EVOLUTION

SCHEMA_DEMO_TABLE = (
    "retail_demo.gold.fact_orders_schema_evolution_demo"
)
# Verify evolved schema

evolved_schema = spark.table(
    SCHEMA_DEMO_TABLE
).schema

column_names = [
    field.name
    for field in evolved_schema.fields
]

print("coupon_code present:", "coupon_code" in column_names)

print("\nEvolved schema:")

spark.table(
    SCHEMA_DEMO_TABLE
).printSchema()

# Check Delta history

print("\nDelta history:")
display(
    spark.sql(f"""
        DESCRIBE HISTORY {SCHEMA_DEMO_TABLE}
    """).select(
        "version",
        "timestamp",
        "operation",
        "operationMetrics"
    )
)

# Final row count

print("\nFinal row count:")
display(
    spark.sql(f"""
        SELECT
            COUNT(*) AS total_rows
        FROM {SCHEMA_DEMO_TABLE}
    """)
)

coupon_code present: True

Evolved schema:
root
 |-- order_id: string (nullable = true)
 |-- order_ts: timestamp (nullable = true)
 |-- order_date: date (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_sk: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_sk: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: decimal(18,2) (nullable = true)
 |-- discount_pct: double (nullable = true)
 |-- gross_amount: decimal(18,2) (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- coupon_code: string (nullable = true)


Delta history:


version,timestamp,operation,operationMetrics
7,2026-08-13T03:12:53.000Z,WRITE,"Map(numFiles -> 1, numOutputRows -> 20, numOutputBytes -> 6703)"
6,2026-08-13T03:12:49.000Z,CREATE OR REPLACE TABLE AS SELECT,"Map(numFiles -> 1, numRemovedFiles -> 2, numRemovedBytes -> 20496, numDeletionVectorsRemoved -> 0, numOutputRows -> 100, numOutputBytes -> 13782)"
5,2026-08-13T02:34:04.000Z,WRITE,"Map(numFiles -> 1, numOutputRows -> 20, numOutputBytes -> 6704)"
4,2026-08-13T02:34:01.000Z,CREATE OR REPLACE TABLE AS SELECT,"Map(numFiles -> 1, numRemovedFiles -> 2, numRemovedBytes -> 20492, numDeletionVectorsRemoved -> 0, numOutputRows -> 100, numOutputBytes -> 13792)"
3,2026-08-12T20:17:24.000Z,WRITE,"Map(numFiles -> 1, numOutputRows -> 20, numOutputBytes -> 6700)"
2,2026-08-12T20:17:20.000Z,CREATE OR REPLACE TABLE AS SELECT,"Map(numFiles -> 1, numRemovedFiles -> 2, numRemovedBytes -> 20496, numDeletionVectorsRemoved -> 0, numOutputRows -> 100, numOutputBytes -> 13792)"
1,2026-08-12T16:07:47.000Z,WRITE,"Map(numFiles -> 1, numOutputRows -> 20, numOutputBytes -> 6704)"
0,2026-08-12T16:07:44.000Z,CREATE OR REPLACE TABLE AS SELECT,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 100, numOutputBytes -> 13792)"



Final row count:


total_rows
120
